# SkinGAN — GAN smoke test on Google Colab

Short run of the **baseline conditional StyleGAN2-ADA** on HAM10000 (option 3): the same pipeline as the
real SkinGAN baseline, with a ~100 kimg budget instead of 1,000.

**Before you start:** `Runtime → Change runtime type → T4 GPU` (or better), then run the cells top to bottom.

| Step | Time on a T4 |
|---|---|
| 1–3 setup | ~2 min |
| 4 dataset (first time: download + export; later: copy from Drive) | ~10 min first time, ~1 min after |
| 5 training, 100 kimg | ~1.5–2 h |
| 6 pictures | ~2 min |

Everything you want to keep (dataset zip, snapshots, pictures) goes to **Google Drive → `MyDrive/SkinGAN/`**,
so a disconnect loses nothing: re-run cells 1–4 and step 5 resumes from the last snapshot.

## 0 · Settings

In [ ]:
REPO_URL  = "https://github.com/AmoghNamuduri/SkinLesion.git"   # our repo (must be pushed & public)
BRANCH    = "main"
DRIVE_DIR = "/content/drive/MyDrive/SkinGAN"                     # persistent storage
KIMG      = 100        # training budget for the test (real run: 1000)
SNAP      = 5          # snapshot every 5 ticks = 20 kimg
RESUME    = True       # continue from the newest snapshot in Drive if there is one

## 1 · Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2 · Google Drive + code

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs(DRIVE_DIR, exist_ok=True)
if not os.path.exists('/content/SkinLesion'):
    !git clone -q -b {BRANCH} {REPO_URL} /content/SkinLesion
else:
    !git -C /content/SkinLesion pull -q
%cd /content/SkinLesion
!pip -q install ninja click

## 3 · StyleGAN2-ADA (NVIDIA) + SkinGAN patches
Clones NVlabs/stylegan2-ada-pytorch into `third_party/` and applies our two marked patches
(domain-loss hook, conditional resume from FFHQ). See `third_party/README.md`.

In [ ]:
!python third_party/setup_stylegan.py

## 4 · Training data: HAM10000 → 256² StyleGAN archive
First time: downloads HAM10000 from Harvard Dataverse (~2.7 GB, no login), makes the **lesion-level split**
(the GAN only sees the train split), and writes the archive. The zip and split CSVs are saved to Drive and reused.

In [ ]:
import shutil, glob
ZIP_NAME   = "ham10000_256_7class.zip"
DRIVE_ZIP  = f"{DRIVE_DIR}/data/{ZIP_NAME}"
LOCAL_ZIP  = f"/content/data/{ZIP_NAME}"
SPLITS_DIR = f"{DRIVE_DIR}/data/splits"
os.makedirs("/content/data", exist_ok=True)

if os.path.exists(DRIVE_ZIP):
    print("using archive from Drive"); shutil.copy(DRIVE_ZIP, LOCAL_ZIP)
else:
    !python scripts/download_data.py --set paths.raw_dir=/content/ham_raw
    !python scripts/export_gan_dataset.py --set paths.raw_dir=/content/ham_raw paths.gan_dataset={LOCAL_ZIP} paths.splits_dir={SPLITS_DIR}
    os.makedirs(os.path.dirname(DRIVE_ZIP), exist_ok=True)
    shutil.copy(LOCAL_ZIP, DRIVE_ZIP)
    shutil.copy("/content/data/real_samples.png", f"{DRIVE_DIR}/data/real_samples.png")
    shutil.rmtree("/content/ham_raw", ignore_errors=True)   # free disk space
print(round(os.path.getsize(LOCAL_ZIP) / 1e6), "MB")

In [ ]:
# what the GAN has to imitate: real training images per class (count in brackets)
from IPython.display import Image, display
display(Image(f"{DRIVE_DIR}/data/real_samples.png", width=900))

## 5 · Train (baseline conditional StyleGAN2-ADA)
Transfer learning from NVIDIA's FFHQ-256 model (`--resume=ffhq256`), ADA augmentation, conditional on the
7 classes. Every 20 kimg StyleGAN writes `fakes0000XX.png` and `network-snapshot-0000XX.pkl` to Drive.
Leave this cell running. You can open the newest `fakes*.png` in Drive while it trains.
Expect the first grids to look like faces; they turn into skin and lesions as training goes on.

*Resuming:* a resumed run restarts StyleGAN's kimg counter from 0, so it trains another `KIMG`.

In [ ]:
RUNS = f"{DRIVE_DIR}/runs/baseline_test"
snaps = sorted(glob.glob(f"{RUNS}/*/network-snapshot-*.pkl"))
resume = f"gan.resume={snaps[-1]}" if (RESUME and snaps) else ""
print("resuming from", snaps[-1] if resume else "FFHQ-256 (transfer learning)")
!python scripts/train_gan.py --config gan_test/test_config.yaml --set paths.gan_dataset={LOCAL_ZIP} paths.gan_runs={RUNS} gan.kimg={KIMG} gan.snap={SNAP} {resume}

## 6 · Pictures
`progression.png`: the same samples at every snapshot · `real_vs_generated.png`: per class, 4 real | 4 generated ·
`class_grid.png`: 8 generated per class. All three are saved in the run folder on Drive.

In [ ]:
!python gan_test/show_results.py --runs {RUNS} --data {LOCAL_ZIP}
run = sorted(d for d in glob.glob(f"{RUNS}/*") if glob.glob(f"{d}/network-snapshot-*.pkl"))[-1]
for name in ["progression.png", "real_vs_generated.png", "class_grid.png"]:
    print(name); display(Image(f"{run}/{name}", width=1000))

## Optional · quick FID (~10 min)
Lower is better; on 100 kimg expect a high number. This is only to check that the metric pipeline works.

In [ ]:
latest = sorted(glob.glob(f"{run}/network-snapshot-*.pkl"))[-1]
!cd third_party/stylegan2-ada-pytorch && python calc_metrics.py --metrics=fid10k_full --network={latest} --data={LOCAL_ZIP} --mirror=1